# 🚀 Training YOLOv8: Deteksi Koin & Tutup Botol (Anti-False-Positive)

Notebook ini dirancang untuk melatih model deteksi objek **Koin** dan **Tutup Botol** secara menyeluruh hingga selesai, lengkap dengan **sampel negatif (wajah & latar)** agar AI tidak lagi keliru mendeteksi wajah manusia sebagai tutup botol.

### ⚠️ Pengaturan Wajib Sebelum Memulai:
1. Di menu atas Colab, klik **Runtime** > **Change runtime type** (Ubah jenis runtime).
2. Pada *Hardware accelerator*, pilih **T4 GPU** lalu klik **Save**.
3. Pastikan file `sortir-yolo.zip` sudah di-upload ke Google Drive utama kamu (`My Drive`).

## 1. Cek Ketersediaan GPU
Pastikan GPU NVIDIA (T4) terdeteksi aktif.

In [ ]:
!nvidia-smi

## 2. Hubungkan ke Google Drive & Ekstrak Proyek
Mengambil file `sortir-yolo.zip` dari Google Drive kamu dan mengekstraknya di Colab.

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# Hapus direktori lama jika ada, lalu ekstrak fresh
!rm -rf /content/sortir-yolo
!unzip -q -o /content/drive/MyDrive/sortir-yolo.zip -d /content

# Masuk ke folder kerja
%cd /content/sortir-yolo
!pwd
!ls -la

## 3. Install Ultralytics (YOLOv8) & Dependensi

In [ ]:
!pip install -q ultralytics pyyaml opencv-python Pillow requests

## 4. Periksa & Kalibrasi Dataset (`data.yaml`)
Menyelaraskan path dataset agar sesuai dengan struktur direktori Google Colab (`/content/sortir-yolo/dataset_gabungan`).

In [ ]:
import os, yaml
from pathlib import Path

yaml_file = Path('/content/sortir-yolo/dataset_gabungan/data.yaml')

if not yaml_file.exists():
    print('⚠️ dataset_gabungan belum ada di dalam zip, menjalankan pembuatan ulang...')
    !python 03_merge_datasets.py

# Kalibrasi path mutlak ke lingkungan Colab
with open(yaml_file, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = '/content/sortir-yolo/dataset_gabungan'
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
if 'test' in cfg:
    cfg['test'] = 'test/images'
cfg['names'] = {0: 'koin', 1: 'tutup_botol'}

with open(yaml_file, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('✅ data.yaml berhasil dikalibrasi:')
print(open(yaml_file).read())

## 5. 🛡️ Tambahkan Negative Samples (Wajah Manusia & Latar Ruangan)
**Ini adalah kunci agar AI tidak mendeteksi wajah sebagai tutup botol.**
Kita menyuntikkan foto wajah dan latar kosong dengan label kosong (*empty label*). Sesuai standar YOLO, gambar tanpa label mengajarkan model untuk tidak menghasilkan *False Positive* pada wajah manusia atau perabot.

In [ ]:
import os, urllib.request
from pathlib import Path

# Contoh sampel wajah publik & ruangan bebas hak cipta (Wikimedia / Raw Sample)
sample_negatives = [
    'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a2/Face-smile.svg/600px-Face-smile.svg.png',
    'https://images.unsplash.com/photo-1544005313-94ddf0286df2?w=640&q=80',
    'https://images.unsplash.com/photo-1507003211169-0a1dd7228f2d?w=640&q=80',
    'https://images.unsplash.com/photo-1500648767791-00dcc994a43e?w=640&q=80',
    'https://images.unsplash.com/photo-1494790108377-be9c29b29330?w=640&q=80',
    'https://images.unsplash.com/photo-1522075469751-3a6694fb2f61?w=640&q=80',
    'https://images.unsplash.com/photo-1534528741775-53994a69daeb?w=640&q=80',
    'https://images.unsplash.com/photo-1517841905240-472988babdf9?w=640&q=80',
    'https://images.unsplash.com/photo-1539571696357-5a69c17a67c6?w=640&q=80',
    'https://images.unsplash.com/photo-1524504388940-b1c1722653e1?w=640&q=80',
    'https://images.unsplash.com/photo-1586023492125-27b2c045efd7?w=640&q=80',  # Meja & Kursi
    'https://images.unsplash.com/photo-1513694203232-719a280e022f?w=640&q=80',  # Ruangan dinding
    'https://images.unsplash.com/photo-1527443224154-c4a3942d3acf?w=640&q=80',  # Meja kosong
]

train_img_dir = Path('/content/sortir-yolo/dataset_gabungan/train/images')
train_lbl_dir = Path('/content/sortir-yolo/dataset_gabungan/train/labels')
val_img_dir   = Path('/content/sortir-yolo/dataset_gabungan/valid/images')
val_lbl_dir   = Path('/content/sortir-yolo/dataset_gabungan/valid/labels')

train_img_dir.mkdir(parents=True, exist_ok=True)
train_lbl_dir.mkdir(parents=True, exist_ok=True)
val_img_dir.mkdir(parents=True, exist_ok=True)
val_lbl_dir.mkdir(parents=True, exist_ok=True)

count = 0
for i, url in enumerate(sample_negatives):
    try:
        target_split = 'valid' if i % 5 == 0 else 'train'
        img_dest = (val_img_dir if target_split == 'valid' else train_img_dir) / f'negative_bg_{i:03d}.jpg'
        lbl_dest = (val_lbl_dir if target_split == 'valid' else train_lbl_dir) / f'negative_bg_{i:03d}.txt'

        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=5) as resp, open(img_dest, 'wb') as out_f:
            out_f.write(resp.read())

        # Buat file label kosong (0 byte) -> standar YOLO untuk latar negatif
        lbl_dest.write_text('')
        count += 1
    except Exception as e:
        pass

print(f'✅ Berhasil menambahkan {count} sampel negatif (wajah & latar kosong) ke dataset!')

## 6. 🔥 Training YOLOv8 (Full Sampai Selesai)
- Target: **80 Epochs** (waktu latih sekitar 10-15 menit di GPU T4).
- `patience=40`: Mencegah training berhenti terlalu cepat sebelum model benar-benar matang.
- `save_period=5`: Setiap 5 epoch tersimpan otomatis ke Google Drive (`MyDrive/yolo_runs/sortir`).
- Model terbaik otomatis disimpan ke file `best.pt`.

In [ ]:
from pathlib import Path
import os, shutil
from ultralytics import YOLO

# Pastikan direktori Google Drive siap untuk menyimpan model
save_dir = Path('/content/drive/MyDrive/yolo_runs')
save_dir.mkdir(parents=True, exist_ok=True)

# Mulai dari bobot pretrained YOLOv8n
model = YOLO('yolov8n.pt')

print('🚀 Memulai proses training YOLOv8...')
results = model.train(
    data='/content/sortir-yolo/dataset_gabungan/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    device=0,             # GPU T4 Colab
    project='/content/drive/MyDrive/yolo_runs',
    name='sortir',
    exist_ok=True,
    patience=40,
    workers=4,
    save_period=5,
    plots=True,
)

# Validasi hasil terbaik
best_pt = Path('/content/drive/MyDrive/yolo_runs/sortir/weights/best.pt')
if best_pt.is_file():
    print('\n📊 Evaluasi Model Terbaik:')
    metrics = YOLO(str(best_pt)).val(data='/content/sortir-yolo/dataset_gabungan/data.yaml', device=0)
    print(f'mAP50    : {metrics.box.map50:.4f}')
    print(f'mAP50-95 : {metrics.box.map:.4f}')
    
    # Salin ke folder lokal proyek models/
    Path('/content/sortir-yolo/models').mkdir(parents=True, exist_ok=True)
    shutil.copy2(best_pt, '/content/sortir-yolo/models/best.pt')
    print('✅ Model terbaik berhasil disimpan di Google Drive & models/best.pt!')

## 7. (Opsional) Lanjutkan Training Jika Terputus
*Jalankan sel ini HANYA JIKA koneksi Colab kemarin terputus di tengah jalan saat training belum selesai.*

In [ ]:
# Hapus tanda pagar di bawah ini jika ingin me-resume training:
# from ultralytics import YOLO
# last_pt = '/content/drive/MyDrive/yolo_runs/sortir/weights/last.pt'
# model = YOLO(last_pt)
# model.train(resume=True)

## 8. Tampilkan Grafik Metrik & Akurasi

In [ ]:
from IPython.display import Image, display
import os

base = '/content/drive/MyDrive/yolo_runs/sortir'
grafik = ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']

for g in grafik:
    p = os.path.join(base, g)
    if os.path.exists(p):
        print(f'--- {g} ---')
        display(Image(p))
    else:
        print(f'{g} belum ada.')

## 9. Unduh File `best.pt` ke Laptop
Klik sel ini untuk langsung mendownload model hasil latih terbaru ke laptop Anda.

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/yolo_runs/sortir/weights/best.pt')